# Crash Course: Math Review and PyTorch Basics

This notebook reviews the math you need for deep learning and shows you how to write it in `PyTorch`. The goal is to refresh and prepare you for the next tasks which will familiarize you with training various types Neural Networks with `PyTorch`. This notebook will be vital for your understanding of subsequent tasks as well as preparing you for actual research work. Knowledge of these skills will be a requirement for you to do research.  

### How to use this notebook
This notebook contains two kinds of questions:

- **Predict:** before running the cell, fill in `my_guess` (or `guess_1`, `guess_2`, ...) with what you think the answer will be. Usually it is a shape, like `(16, 3)`. Then run the cell to check. Wrong guesses are fine, this will be checked for completion. We hope to build your intuition through theses questions. 
- **Exercise:** replace each `None` with your own code. This will be check for correctness. After writing your code run the test to verify your solution is correct. 

Run the cells from top to bottom, because later cells use variables from earlier ones.

### Contents
1. Tensor basics
2. Indexing and shape manipulation
3. Math review
4. Activation functions
5. Saving and loading with `PyTorch`

## Submission
Once you have finished completed this notebook submit it as your delieverable through canvas. 

# 0. Setup
The library is called `PyTorch`, but in Python you import it as `torch`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)  # makes the "random" numbers the same every time you run the notebook

In [ ]:
# Helper functions: run this cell, but you don't need to read it.

def fm(matrix, precision=1):
    matrix = np.asarray(matrix)
    assert len(matrix.shape) <= 2
    m = np.atleast_2d(matrix)

    if np.issubdtype(m.dtype, np.floating):
        cells = [[f"{x:.{precision}f}" for x in row] for row in m]
    else:
        cells = [[str(x) for x in row] for row in m]

    widths = [max((len(r[j]) for r in cells), default=0) for j in range(m.shape[1])]
    lines = ["  ".join(c.rjust(w) for c, w in zip(row, widths)) for row in cells]

    n = len(lines)
    if n == 1:
        return f"[ {lines[0]} ]"
    out = []
    for i, line in enumerate(lines):
        l, r = ("⎡", "⎤") if i == 0 else ("⎣", "⎦") if i == n - 1 else ("⎢", "⎥")
        out.append(f"{l} {line} {r}")
    return "\n".join(out)


def check_prediction(guess, actual):
    if isinstance(actual, torch.Size):
        actual = tuple(actual)
    if isinstance(guess, list):
        guess = tuple(guess)
    if guess is None:
        print("Fill in your guess first, then re-run this cell.")
        return
    both_numbers = isinstance(guess, (int, float)) and isinstance(actual, (int, float)) \
        and not isinstance(guess, bool) and not isinstance(actual, bool)
    correct = abs(guess - actual) < 1e-4 if both_numbers else guess == actual
    if correct:
        print(f"Correct! The answer is {actual}")
    else:
        print(f"Not quite. You guessed {guess}, but the answer is {actual}")

# 1. Tensor basics
A **tensor** is PyTorch's version of an array: a grid of numbers with any number of dimensions.

| Object | Number of dimensions | Example shape |
|---|---|---|
| scalar | 0 | `()` |
| vector | 1 | `(3,)` |
| matrix | 2 | `(2, 3)` |
| batch of color images | 4 | `(16, 3, 32, 32)` |

If you know `NumPy`, tensors work a lot like NumPy arrays, and much of this section will look familiar. Tensors add two things: they can run on a GPU, and they can track gradients.

## 1.1 Creating tensors

In [ ]:
# from a Python list
A = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(fm(A))
print(A.dtype)  # integers in -> an integer tensor (int64)

B = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print(B.dtype)  # neural networks almost always use float32

In [ ]:
shape = (2, 4)  # feel free to play around with this and observe how the matricies change 
# NOTE: the shape can be more than 2 dimensions but for the sake of visualisation we cap it at 2 dimensions

zeros = torch.zeros(shape)
print("A tensor of all zeros:\n", fm(zeros), "\n")

ones = torch.ones(shape)
print("A tensor of all ones:\n", fm(ones), "\n")

rand = torch.rand(shape)
print("Random numbers from a uniform distribution on [0, 1):\n", fm(rand), "\n")

randn = torch.randn(shape)
print("Random numbers from a normal distribution (mean 0, std 1):\n", fm(randn))

**`rand` vs `randn`:** `torch.rand` draws numbers uniformly from `[0, 1)`. `torch.randn` draws from a *standard normal* (bell curve) distribution, so its values can be negative and are sometimes far from 0. `randn` is often used to initialize weights. A histogram of 10,000 samples shows the difference:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(torch.rand(10_000).numpy(), bins=50)
axes[0].set_title("torch.rand: uniform on [0, 1)")
axes[1].hist(torch.randn(10_000).numpy(), bins=50)
axes[1].set_title("torch.randn: normal, mean 0, std 1")
plt.show()

In [ ]:
# arange(start, end, step) counts from start up to, but NOT including, end
arange = torch.arange(0, 10, 1)
print("arange(0, 10, 1):\n", fm(arange), "\n")
print("arange(5) is a shortcut for arange(0, 5, 1):\n", fm(torch.arange(5)))

In [ ]:
# zeros_like / ones_like / rand_like make a new tensor with the SAME shape and dtype as an existing one
template = torch.randn(3, 5)
z = torch.zeros_like(template)
print(z.shape, z.dtype)

## 1.2 Inspecting tensors
Use `.shape`, `.ndim`, `.numel()`, and `.dtype` to inspect a tensor. When code breaks, print `.shape` first. Most deep learning bugs are shape bugs.

We'll use a batch of images with shape `[16, 3, 32, 32]` throughout this notebook, meaning **[Batch Size, Channels, Height, Width]**: 16 color (RGB) images, each 32×32 pixels.

In [ ]:
image_batch = torch.rand(16, 3, 32, 32)

# PREDICT: how many dimensions does image_batch have, and how many numbers does it hold in total?
guess_1 = None  # image_batch.ndim
guess_2 = None  # image_batch.numel()   (you can type a math expression)

check_prediction(guess_1, image_batch.ndim)
check_prediction(guess_2, image_batch.numel())

In [ ]:
print("shape:", image_batch.shape)    # size of each dimension
print("ndim: ", image_batch.ndim)     # number of dimensions
print("numel:", image_batch.numel())  # total number of elements
print("dtype:", image_batch.dtype)    # data type of the elements

## 1.3 Converting between NumPy and PyTorch
- `torch.from_numpy(array)` converts NumPy to PyTorch
- `tensor.numpy()` converts PyTorch to NumPy

Watch the dtype. NumPy defaults to `float64`, but PyTorch layers expect `float32`. Use `.float()` to convert.

In [ ]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
back = t.numpy()

print(type(t), t.dtype)        # float64, inherited from NumPy
print(t.float().dtype)         # converted to float32
print(type(back))

`torch.from_numpy` and `.numpy()` do **not** copy the data. The tensor and the array share the same memory.

**Predict:** if we change `arr[0]` after creating `t`, what is `t[0]`?

In [ ]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
arr[0] = 100.0

my_guess = None  # the value of t[0]
check_prediction(my_guess, t[0].item())  # .item() turns a one-element tensor into a plain Python number

If you want an independent copy, use `torch.tensor(arr)` instead. You'll often see `tensor.detach().cpu().numpy()` in real code, because `.numpy()` doesn't work on GPU tensors or on tensors that track gradients.

We will won't go over what that means in this notebook but it will become important later on...
## Exercise 1

In [ ]:
# 1. A float32 tensor of zeros with shape (4, 5)
ex_zeros = None

# 2. The even numbers 0, 2, 4, ..., 18 using torch.arange
evens = None

# 3. A tensor of ones with the same shape and dtype as image_batch (don't type the shape by hand!)
ex_ones = None

# 4. Convert this NumPy array into a float32 tensor
np_data = np.arange(12).reshape(3, 4)
from_np = None

# 5. The total number of elements in from_np, using a tensor method
n_elements = None

In [ ]:
# Check your work
assert ex_zeros.shape == (4, 5) and ex_zeros.dtype == torch.float32 and (ex_zeros == 0).all()
assert torch.equal(evens, torch.tensor([0, 2, 4, 6, 8, 10, 12, 14, 16, 18]))
assert ex_ones.shape == image_batch.shape and ex_ones.dtype == image_batch.dtype and (ex_ones == 1).all()
assert from_np.dtype == torch.float32 and from_np.shape == (3, 4)
assert n_elements == 12
print("All checks passed")

# 2. Indexing and shape manipulation
A tensor's dimensions come in a set order. Keeping track of what each dimension means is extremely important for deep learning. Remember, `image_batch` is `[Batch, Channels, Height, Width]`.

## 2.1 Indexing

In [ ]:
# Let's get the 15th image from the batch. Tensors are 0-indexed, so the 15th image is at index 14.
fifteenth_image = image_batch[14]
print(fifteenth_image.shape)

Indexing with a single integer **removed** the batch dimension. That is often fine, but models expect a batch dimension even when the batch holds only one image. Here are two ways to keep it:

In [ ]:
# option 1: index, then add the dimension back with unsqueeze
res1 = image_batch[14].unsqueeze(0) # in this case we are adding a dimension to the 0th dimension (i.e. inserting at the start)

# option 2: slice instead of index. Slicing always keeps the dimension.
res2 = image_batch[14:15] 

print(res1.shape)
assert torch.equal(res1, res2)

In [ ]:
# We can index into several dimensions at once.
# Say we want only the first channel of the last image in the batch:
res1 = image_batch[-1, 0, :, :]
print(res1.shape)

# To keep all four dimensions, use slices. -1: means "from the last element to the end".
res2 = image_batch[-1:, 0:1, :, :]
print(res2.shape)
assert torch.equal(res2.squeeze(), res1)

In [ ]:
# PREDICT the shape of each expression, then run the cell.
guess_1 = None  # image_batch[0, 0]
guess_2 = None  # image_batch[:, 0]
guess_3 = None  # image_batch[:8, :, :16, :16]
guess_4 = None  # image_batch[-1:]

check_prediction(guess_1, image_batch[0, 0].shape)
check_prediction(guess_2, image_batch[:, 0].shape)
check_prediction(guess_3, image_batch[:8, :, :16, :16].shape)
check_prediction(guess_4, image_batch[-1:].shape)

## 2.2 `squeeze` and `unsqueeze`
- `unsqueeze(dim)` **inserts** a dimension of size 1 at position `dim`.
- `squeeze(dim)` **removes** dimension `dim` only if its size is 1.
- `squeeze()` with no argument removes *every* size-1 dimension. This can be convenient but risky, be certain you know the dimensions of your tensors or else this may lead to silent errors. Prefer passing `dim`.

In [ ]:
x = torch.rand(3, 32, 32)
y = torch.rand(1, 3, 1, 32)

# PREDICT
guess_1 = None  # x.unsqueeze(0)
guess_2 = None  # x.unsqueeze(-1)
guess_3 = None  # y.squeeze()
guess_4 = None  # y.squeeze(0)

check_prediction(guess_1, x.unsqueeze(0).shape)
check_prediction(guess_2, x.unsqueeze(-1).shape)
check_prediction(guess_3, y.squeeze().shape)
check_prediction(guess_4, y.squeeze(0).shape)

## 2.3 Re-ordering dimensions with `permute`

In [ ]:
# Our image batch has 4 dimensions, [batch, channels, height, width], numbered [0, 1, 2, 3].
# To re-order them to [batch, height, width, channels], list the old dimension numbers in the new order:
re_ordered = image_batch.permute(0, 2, 3, 1)  # move the channel dimension to the back
print(re_ordered.shape)

### Exercise 2a
Let's put this to work. We want to visualize one of our images with `matplotlib`. It can only show one image at a time, and it expects the channel dimension to be **last**.

**Note:** `image_batch` is filled with random numbers, so the picture will look like colorful random noise.

In [ ]:
image = None  # get the first image from the batch: shape [3, 32, 32]
image_re_ordered = None  # re-order the dimensions so matplotlib can plot it

assert image_re_ordered.shape == (32, 32, 3)
plt.imshow(image_re_ordered)
plt.title("Random noise")
plt.show()

## 2.4 `reshape` vs `view`
Both change a tensor's shape without changing its data, as long as the total number of elements stays the same. You can pass `-1` for **one** dimension, and PyTorch will work out its size.

In [ ]:
x = torch.arange(12)
print(fm(x), "\n")
print(fm(x.reshape(3, 4)), "\n")
print(fm(x.view(4, 3)), "\n")
print(x.reshape(2, -1).shape)  # -1 means "whatever makes the numbers work": 12 / 2 = 6

In [ ]:
# PREDICT
guess_1 = None  # torch.arange(12).reshape(3, -1)
guess_2 = None  # torch.arange(24).reshape(2, 3, -1)
guess_3 = None  # torch.arange(12).reshape(3, 4).reshape(-1)    (hint: a 1-D shape is written (n,))

check_prediction(guess_1, torch.arange(12).reshape(3, -1).shape)
check_prediction(guess_2, torch.arange(24).reshape(2, 3, -1).shape)
check_prediction(guess_3, torch.arange(12).reshape(3, 4).reshape(-1).shape)

# Some reshapes are impossible: 12 elements can't be split into rows of 5
try:
    torch.arange(12).reshape(5, -1)
except RuntimeError as e:
    print("\nRuntimeError:", e)

**So what's the difference?** `view` never copies data, so it only works when the tensor is laid out contiguously in memory. Operations like `permute` and `.T` return non-contiguous tensors, and `view` fails on them. `reshape` always works because it copies when it has to. To be safe you should default to `reshape`.

In [ ]:
permuted = image_batch.permute(0, 2, 3, 1)
print("contiguous?", permuted.is_contiguous())

try:
    permuted.view(16, -1)
except RuntimeError as e:
    print("view failed:", e)

print("reshape works:", permuted.reshape(16, -1).shape)

## 2.5 `flatten`
Linear (fully connected) layers expect each sample to be a 1-D vector of features. To feed images into one, flatten everything **except** the batch dimension: `[16, 3, 32, 32]` → `[16, 3072]`.

In [ ]:
flat1 = image_batch.flatten(start_dim=1)                 # flatten dims 1 onward, keep dim 0 (batch)
flat2 = image_batch.reshape(16, -1)                      # the same thing with reshape
flat3 = image_batch.reshape(image_batch.shape[0], -1)    # the same, without hard-coding the batch size

print(flat1.shape)
assert torch.equal(flat1, flat2) and torch.equal(flat2, flat3)

# without start_dim, flatten flattens EVERYTHING, including the batch dimension
print(image_batch.flatten().shape)

## 2.6 `cat` vs `stack`
- `torch.cat` joins tensors along an **existing** dimension. All other dimensions must match.
- `torch.stack` creates a **new** dimension and places the tensors along it. All shapes must match exactly.

In [ ]:
a = torch.zeros(16, 3)
b = torch.ones(16, 3)

#  PREDICT
guess_1 = None  # torch.cat([a, b], dim=0)
guess_2 = None  # torch.cat([a, b], dim=1)
guess_3 = None  # torch.stack([a, b], dim=0)
guess_4 = None  # torch.stack([a, b], dim=1)

check_prediction(guess_1, torch.cat([a, b], dim=0).shape)
check_prediction(guess_2, torch.cat([a, b], dim=1).shape)
check_prediction(guess_3, torch.stack([a, b], dim=0).shape)
check_prediction(guess_4, torch.stack([a, b], dim=1).shape)

A common use for each:
- `cat`: combining two batches into one bigger batch.
- `stack`: turning a list of individual images, each `[3, 32, 32]`, into a batch.

### Exercise 2b

In [ ]:
# A list of 8 separate images, like you might get from loading image files one at a time
images = [torch.rand(3, 32, 32) for _ in range(8)]

# 1. Combine the list into one batch with shape [8, 3, 32, 32]
batch = None

# 2. Flatten each image in `batch`, giving shape [8, 3072]
flat = None

# 3. Take the first image from `batch` but keep the batch dimension: shape [1, 3, 32, 32]
first = None

# 4. Join `batch` with itself along the batch dimension: shape [16, 3, 32, 32]
doubled = None

# 5. Convert `batch` to channels-last: shape [8, 32, 32, 3]
channels_last = None

In [ ]:
# Check your work
assert batch.shape == (8, 3, 32, 32) and torch.equal(batch[3], images[3])
assert flat.shape == (8, 3072) and torch.equal(flat[2], images[2].flatten())
assert first.shape == (1, 3, 32, 32) and torch.equal(first[0], images[0])
assert doubled.shape == (16, 3, 32, 32) and torch.equal(doubled[8], images[0])
assert channels_last.shape == (8, 32, 32, 3) and torch.equal(channels_last[5, :, :, 1], images[5][1])
print("All checks passed")

# 3. Math review
Neural networks are built from a small set of operations. This section reviews each one in math notation and in PyTorch.

## 3.1 Element-wise vs matrix multiplication
- **Element-wise** (`*`): multiply matching entries. $(A \odot B)_{ij} = A_{ij} \, B_{ij}$
- **Matrix multiplication** (`@`): $(AB)_{ij} = \sum_k A_{ik} B_{kj}$. Entry $(i, j)$ is the dot product of **row $i$ of $A$** with **column $j$ of $B$**.

In [ ]:
A = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
B = torch.tensor([[3, 4], [1, 2]], dtype=torch.float32)
print("A:\n", fm(A), "\n")
print("B:\n", fm(B), "\n")

# PREDICT: work out the top-left entry of A @ B by hand (row 0 of A, column 0 of B)
my_guess = None
check_prediction(my_guess, (A @ B)[0, 0].item())

In [ ]:
print("A * B (element-wise):\n", fm(A * B), "\n")
print("A @ B (matrix multiplication):\n", fm(A @ B))

In [ ]:
# There are several equivalent ways to multiply matrices
mul1 = A @ B
mul2 = torch.matmul(A, B)
mul3 = A.mm(B)  # mm only works on 2-D tensors
assert torch.equal(mul1, mul2) and torch.equal(mul2, mul3)

# and to add them
add1 = A + B
add2 = torch.add(A, B)
add3 = A.add(B)
assert torch.equal(add1, add2) and torch.equal(add2, add3)

Order matters for matrix multiplication but not for element-wise multiplication:

In [ ]:
print("A @ B:\n", fm(A @ B), "\n")
print("B @ A:\n", fm(B @ A), "\n")
print("A @ B == B @ A?", torch.equal(A @ B, B @ A))
print("A * B == B * A?", torch.equal(A * B, B * A))

## 3.2 The matmul shape rule
$$(m \times n) \;@\; (n \times p) \;\rightarrow\; (m \times p)$$

The **inner** dimensions must match, and they disappear. The **outer** dimensions are kept. A quick way to check is to write the shapes side by side: (2 × **3**) @ (**3** × 4) → (2 × 4).

In [ ]:
X = torch.rand(5, 4)
W = torch.rand(4, 3)
v = torch.rand(3)

# PREDICT
guess_1 = None  # X @ W
guess_2 = None  # W.T @ X.T
guess_3 = None  # X @ W @ v      (a 1-D tensor acts like a vector; a 1-D shape is written (n,))

check_prediction(guess_1, (X @ W).shape)
check_prediction(guess_2, (W.T @ X.T).shape)
check_prediction(guess_3, (X @ W @ v).shape)

In [ ]:
# What happens when the inner dimensions DON'T match? This error is deliberate.
try:
    X @ X  # (5 x 4) @ (5 x 4): the inner dimensions are 4 and 5
except RuntimeError as e:
    print("RuntimeError:", e)

Read that error carefully: you will see it many times. The fix is to make sure the dimensions line up correctly.

## 3.3 Transpose
Transposing swaps rows and columns: $(A^T)_{ij} = A_{ji}$.
- `.T` works on 2-D tensors.
- `.transpose(dim0, dim1)` swaps any two dimensions of a tensor with any number of dimensions.

In [ ]:
A = torch.tensor([[1, 2, 3], [4, 5, 6]])
A_t = A.T

print("The original tensor:\n", fm(A), "\n")
print("The transposed tensor:\n", fm(A_t))
assert torch.equal(A.T, A.transpose(0, 1))

In [ ]:
# PREDICT (remember image_batch is [16, 3, 32, 32])
guess_1 = None  # image_batch.transpose(1, 3)
guess_2 = None  # image_batch.transpose(2, 3)

check_prediction(guess_1, image_batch.transpose(1, 3).shape)
check_prediction(guess_2, image_batch.transpose(2, 3).shape)

The second shape didn't change because height and width are both 32, but the data did change. Each image is now flipped along its diagonal.

## 3.4 Dot product
$$a \cdot b = \sum_i a_i b_i$$
Multiply element-wise, then sum. The dot product measures how well two vectors line up. Two facts to remember:
- Every entry of a matrix multiplication is a dot product (see 3.1).
- Every output of a linear layer is a dot product of the input with a weight vector, plus a bias (see 3.9).

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

#  PREDICT: compute a · b in your head
my_guess = None
check_prediction(my_guess, torch.dot(a, b).item())

# three equivalent ways
print(torch.dot(a, b), (a * b).sum(), a @ b)

### Exercise 3a: Matrix multiplication from dot products
Implement matrix multiplication using **only** `torch.dot`, without `@`. Use the definition from 3.1: entry `(i, j)` is the dot product of row `i` of `A` and column `j` of `B`.

In [ ]:
def my_matmul(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"inner dimensions don't match: {n} vs {n2}"
    C = torch.zeros(m, p)
    for i in range(m):
        for j in range(p):
            C[i, j] = None
    return C

In [ ]:
# Check your work
P = torch.randn(3, 4)
Q = torch.randn(4, 2)
assert my_matmul(P, Q).shape == (3, 2)
assert torch.allclose(my_matmul(P, Q), P @ Q, atol=1e-6)  # allclose allows for tiny floating-point rounding differences
print("All checks passed")

## 3.5 Broadcasting
When an element-wise operation gets two tensors of different shapes, PyTorch tries to **broadcast** them. It stretches dimensions of size 1 (or missing dimensions) so the shapes match. The rule:

1. Line the shapes up **from the right**.
2. Each pair of dimensions must be **equal**, or one of them must be **1** (a missing dimension counts as 1).

| | shape |
|---|---|
| `x` | 16 × 3 |
| `bias` | &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3 |
| `x + bias` | 16 × 3 |

The most common use in deep learning is adding a bias vector to every sample in a batch:

In [ ]:
x = torch.zeros(16, 3)                # a batch of 16 samples with 3 features each
bias = torch.tensor([1.0, 2.0, 3.0])  # one bias value per feature
out = x + bias                        # bias is added to every row

print(out.shape)
print(fm(out[:4]))  # the first 4 rows

In [ ]:
x = torch.zeros(16, 3)

#  PREDICT the shape of each result
guess_1 = None  # x + torch.ones(3)
guess_2 = None  # x + torch.ones(16, 1)
guess_3 = None  # torch.ones(16, 1) + torch.ones(1, 3)
guess_4 = None  # torch.ones(4, 1, 5) + torch.ones(3, 1)

check_prediction(guess_1, (x + torch.ones(3)).shape)
check_prediction(guess_2, (x + torch.ones(16, 1)).shape)
check_prediction(guess_3, (torch.ones(16, 1) + torch.ones(1, 3)).shape)
check_prediction(guess_4, (torch.ones(4, 1, 5) + torch.ones(3, 1)).shape)

In [ ]:
# A deliberate broadcasting error: the rightmost dimensions are 3 and 16, and neither is 1
try:
    x + torch.ones(16)
except RuntimeError as e:
    print("RuntimeError:", e)

# To add one value per SAMPLE, give it shape [16, 1] instead
print((x + torch.ones(16).unsqueeze(1)).shape)

## 3.6 Reductions: `sum`, `mean`, `max`, `argmax`
A **reduction** collapses a dimension by combining its values. The `dim=` argument says which dimension to collapse, and that dimension **disappears** from the shape.

This is easy to mix up: `A.sum(dim=1)` on a 2-D tensor collapses the columns, so you get **one value per row**.

In [ ]:
A = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print(fm(A), "\n")

row_sum = A.sum(dim=1)  # collapse dim 1 (columns) -> one sum per row
col_sum = A.sum(dim=0)  # collapse dim 0 (rows)    -> one sum per column

print("Sum of each row:\n", fm(row_sum), "\n")
print("Sum of each column:\n", fm(col_sum), "\n")
print("Mean of each row:\n", fm(A.mean(dim=1)), "\n")
print("Sum of everything:", A.sum())

In [ ]:
# max(dim=...) returns both the max values and their indices; argmax returns only the indices
row_max_value = A.max(dim=1).values
row_max_idx = A.argmax(dim=1)
col_max_value = A.max(dim=0).values
col_max_idx = A.argmax(dim=0)

print("Max value in each row:\n", fm(row_max_value), "\n")
print("Index of the max value in each row:\n", fm(row_max_idx), "\n")
print("Max value in each column:\n", fm(col_max_value), "\n")
print("Index of the max value in each column:\n", fm(col_max_idx))

**`keepdim=True`** keeps the collapsed dimension with size 1 instead of removing it. That way the result still broadcasts against the original tensor. For example, here is how to divide each row by its own sum:

In [ ]:
print(A.sum(dim=1).shape)                # torch.Size([2])
print(A.sum(dim=1, keepdim=True).shape)  # torch.Size([2, 1])

row_sums = A.sum(dim=1, keepdim=True)    # [2, 1]
normalized = A / row_sums                # [2, 3] / [2, 1] -> broadcasts
print(fm(normalized, precision=3))
print("each row now sums to:", normalized.sum(dim=1))

# Without keepdim the shapes are [2, 3] and [2], so the rightmost dimensions are 3 vs 2: error!
try:
    A / A.sum(dim=1)
except RuntimeError as e:
    print("RuntimeError:", e)

 **Predict.** Hint: a tensor holding a single number has shape `()`.

In [ ]:
scores = torch.rand(16, 10)  # e.g. 16 images, with a score for each of 10 classes

guess_1 = None  # scores.sum(dim=0)
guess_2 = None  # scores.mean(dim=1)
guess_3 = None  # scores.argmax(dim=1, keepdim=True)
guess_4 = None  # scores.sum()

check_prediction(guess_1, scores.sum(dim=0).shape)
check_prediction(guess_2, scores.mean(dim=1).shape)
check_prediction(guess_3, scores.argmax(dim=1, keepdim=True).shape)
check_prediction(guess_4, scores.sum().shape)

### Exercise 3b: Reductions

In [ ]:
# Model outputs for a batch of 4 samples and 3 classes (a higher score means more confident)
scores = torch.tensor([[2.0, 1.0, 0.1],
                       [0.5, 2.5, 0.3],
                       [1.2, 0.2, 3.1],
                       [0.0, 0.0, 1.0]])
labels = torch.tensor([0, 1, 2, 1])  # the correct class for each sample

# 1. The predicted class for each sample (the index of its highest score). Shape [4]
preds = None

# 2. The fraction of samples where the prediction matches the label, as a Python float.
#    Hint: (preds == labels) gives True/False values. Convert with .float(), take the .mean(), then .item()
accuracy = None

# 3. The average score for each class across the batch. Shape [3]
class_means = None

# 4. Subtract each sample's max score from every score in that sample's row. Shape [4, 3]
#    Hint: you'll need keepdim
shifted = None

In [ ]:
# Check your work
assert torch.equal(preds, torch.tensor([0, 1, 2, 2]))
assert isinstance(accuracy, float) and abs(accuracy - 0.75) < 1e-6
assert torch.allclose(class_means, torch.tensor([0.925, 0.925, 1.125]))
assert shifted.shape == (4, 3) and (shifted.max(dim=1).values == 0).all()
print("All checks passed! 🎉")

## 3.7 `exp` and `log`
- $e^x$ is **always positive** and grows very fast.
- $\log x$ is its inverse. It is only defined for $x > 0$.

Why they matter:
- `exp` turns any score, even a negative one, into a positive number. Softmax (next section) relies on this.
- `log` turns products into sums: $\log(ab) = \log a + \log b$. It also turns tiny probabilities into reasonable-sized negative numbers. The loss functions in the next notebook use $-\log(p)$.

In [ ]:
# PREDICT
guess_1 = None  # torch.exp(torch.tensor(0.0))
guess_2 = None  # torch.log(torch.tensor(1.0))

check_prediction(guess_1, torch.exp(torch.tensor(0.0)).item())
check_prediction(guess_2, torch.log(torch.tensor(1.0)).item())

In [ ]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("x:          ", x)
print("exp(x):     ", torch.exp(x))
print("log(exp(x)):", torch.log(torch.exp(x)))
print()
print("log(0)  =", torch.log(torch.tensor(0.0)))   # -inf
print("log(-1) =", torch.log(torch.tensor(-1.0)))  # nan (not a number)
print()
p = torch.tensor([0.9, 0.8, 0.95, 0.7])
print("log of a product:", torch.log(p.prod()).item())
print("sum of the logs: ", torch.log(p).sum().item())

## 3.8 Softmax: turning scores into probabilities
A classifier outputs one raw score per class. These scores are called **logits**, and they can be any real number. To read them as probabilities, every value must be ≥ 0 and the values must sum to 1. Softmax does exactly that:

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

`exp` makes every value positive, and dividing by the total makes them sum to 1. The largest logit still gets the largest probability.

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(logits, dim=0)
print("probabilities:", probs)
print("sum:", probs.sum())

# For a batch [batch, classes], apply softmax over the classes (dim=1), so each ROW sums to 1
batch_logits = torch.randn(4, 3)
print(torch.softmax(batch_logits, dim=1).sum(dim=1))

In [ ]:
# PREDICT: if we add 100 to every logit, do the probabilities change? (True or False)
my_guess = None
changed = not torch.allclose(torch.softmax(logits, dim=0), torch.softmax(logits + 100, dim=0))
check_prediction(my_guess, changed)

Softmax only depends on the **differences** between logits. That's why subtracting each row's max, as in Exercise 3b #4, doesn't change the result. It is still worth doing, because `exp` of a large number overflows to `inf`:

In [ ]:
big = torch.tensor([1000.0, 999.0])
naive = torch.exp(big) / torch.exp(big).sum()
print("naive softmax:", naive)                         # inf / inf = nan
print("torch.softmax:", torch.softmax(big, dim=0))     # PyTorch subtracts the max for you

### Exercise 3c: Softmax by hand
Implement softmax over the classes (`dim=1`) of a `[batch, classes]` tensor. Use only `max`, `exp`, `sum`, subtraction, and division. Don't call `torch.softmax`!

In [ ]:
def my_softmax(z):
    z = None  # subtract each row's max (for numerical stability)
    exps = None
    return None

In [ ]:
# Check your work
test = torch.randn(5, 4) * 10
out = my_softmax(test)
assert out.shape == (5, 4)
assert torch.allclose(out.sum(dim=1), torch.ones(5)), "each row should sum to 1"
assert torch.allclose(out, torch.softmax(test, dim=1), atol=1e-6)
assert not torch.isnan(my_softmax(torch.tensor([[1000.0, 999.0]]))).any(), "overflow! did you subtract the max?"
print("All checks passed! 🎉")

## 3.9 The linear layer: $y = xW^T + b$
This equation is the core building block of an MLP. For a batch of inputs:

| | meaning | shape |
|---|---|---|
| $x$ | input | `[batch, in_features]` |
| $W$ | weights | `[out_features, in_features]` |
| $b$ | bias | `[out_features]` |
| $y$ | output | `[batch, out_features]` |

$$y = xW^T + b$$

Each output feature $j$ is $y_j = \sum_i x_i W_{ji} + b_j$, the **dot product** of the input with row $j$ of $W$, plus a bias. So one linear layer computes `out_features` dot products at once, for every sample in the batch.

PyTorch stores each output neuron's weights as a **row** of $W$, which is why we need the transpose. Shape check: `[batch, in] @ [in, out]` → `[batch, out]`, and adding `b` of shape `[out]` broadcasts across the batch.

In [ ]:
x = torch.randn(16, 4)
W = torch.randn(3, 4)
b = torch.randn(3)

# PREDICT
guess_1 = None  # W.T
guess_2 = None  # x @ W.T
guess_3 = None  # x @ W.T + b

check_prediction(guess_1, W.T.shape)
check_prediction(guess_2, (x @ W.T).shape)
check_prediction(guess_3, (x @ W.T + b).shape)

## 3.10 Derivatives, gradients, and the chain rule (theory)
**Training** a network means adjusting $W$ and $b$ so that a **loss**, a single number that measures how wrong the network is, gets smaller. To know which way to nudge each parameter, we need derivatives.

**Derivative.** The derivative is the slope: how much the output changes per tiny change in the input.
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$
For example, $f(x) = x^2$ has $f'(x) = 2x$. At $x = 3$ the slope is $6$, so increasing $x$ by a tiny $0.01$ increases $f$ by about $0.06$.

**Gradient.** A loss depends on *many* parameters: $L(w_1, \dots, w_n)$. The **partial derivative** $\frac{\partial L}{\partial w_i}$ is the slope with respect to $w_i$ while every other parameter is held fixed. The **gradient** $\nabla L$ collects all of these into one tensor with the same shape as the parameters. It points in the direction in which $L$ *increases* fastest, so to *decrease* the loss we step the opposite way:
$$w \leftarrow w - \eta \, \nabla_w L$$
Here $\eta$ is the **learning rate**. This update is **gradient descent**, and it is the heart of the next notebook.

**Chain rule.** A network is a chain of functions, like $y = f(g(x))$. To differentiate a chain, multiply the derivatives of the links:
$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}, \qquad \text{where } u = g(x)$$
For example, take $y = (3x+1)^2$. Let $u = 3x + 1$, so $y = u^2$. Then $\frac{dy}{du} = 2u$ and $\frac{du}{dx} = 3$, so $\frac{dy}{dx} = 2(3x+1) \cdot 3 = 6(3x+1)$.

A network's loss looks like $L(\text{linear}_2(\text{relu}(\text{linear}_1(x))))$. Applying the chain rule link by link, from the loss back toward the input, is called **backpropagation**. PyTorch's **autograd** does this for you automatically (next notebook), but knowing what it computes makes debugging much easier.

We can check a derivative numerically by measuring the slope between two very close points:

In [ ]:
def numerical_derivative(f, x, h=1e-4):
    # the slope of f between x - h and x + h
    return (f(x + h) - f(x - h)) / (2 * h)

print(numerical_derivative(lambda x: x ** 2, 3.0))  # should be close to 2 * 3 = 6

### Exercise 3d: Derivatives with pencil and paper
Work these out **by hand** and enter numbers, not code. The check cell compares your answers to `numerical_derivative`.

In [ ]:
# 1. f(x) = x**2. What is f'(3)?
answer_1 = None

# 2. g(x) = (3x + 1)**2. Use the chain rule. What is g'(2)?
answer_2 = None

# 3. s(x) = sigmoid(x) = 1 / (1 + e^(-x)). Its derivative is s(x) * (1 - s(x)). What is s'(0)?
answer_3 = None

# 4. A one-weight "model" predicts w * x. With x = 2 and the true answer y = 10,
#    its squared-error loss is L(w) = (w * 2 - 10)**2. What is dL/dw at w = 3?
answer_4 = None

In [ ]:
# Check your work
import math

checks = [
    (answer_1, lambda x: x ** 2, 3.0),
    (answer_2, lambda x: (3 * x + 1) ** 2, 2.0),
    (answer_3, lambda x: 1 / (1 + math.exp(-x)), 0.0),
    (answer_4, lambda w: (w * 2 - 10) ** 2, 3.0),
]
for i, (answer, f, x) in enumerate(checks, start=1):
    assert answer is not None, f"fill in answer_{i}"
    assert abs(answer - numerical_derivative(f, x)) < 1e-3, f"answer_{i} doesn't match the numerical derivative"
print("All checks passed")

**Think about answer 4.** The gradient is **negative**, so increasing $w$ *lowers* the loss. One gradient descent step with $\eta = 0.1$ gives $w \leftarrow 3 - 0.1 \cdot (-16) = 4.6$. That moves $w$ toward the best value, $w = 5$ (since $5 \cdot 2 = 10$). Training a neural network works the same way, just with thousands or millions of weights at once.

# 4. Activation functions
If you stack linear layers with nothing between them, they collapse into a single linear layer: $W_2(W_1x) = (W_2W_1)x$. That kind of network can only learn linear functions. **Activation functions** add non-linearity between layers, which lets a network learn much more complex functions.

| Function | Formula | Output range | Typical use |
|---|---|---|---|
| ReLU | $\max(0, x)$ | $[0, \infty)$ | the default for hidden layers |
| Sigmoid | $\frac{1}{1 + e^{-x}}$ | $(0, 1)$ | a probability for yes/no outputs |
| Tanh | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | $(-1, 1)$ | hidden layers when zero-centered outputs help |
| Softmax | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | $(0, 1)$, sums to 1 | the final layer of a multi-class classifier |

This section focuses on *using* activation functions. To learn more about why each one exists, see the [PyTorch docs](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity) for the full list.

In [ ]:
A = torch.tensor([[-1, 2], [3, -4]], dtype=torch.float32)

# Three equivalent ways to apply ReLU, which sets negative values to 0 and leaves the rest alone
relu_out = torch.relu(A)       # a plain function
functional_relu = F.relu(A)    # from torch.nn.functional
nn_relu = nn.ReLU()(A)         # as a module (a layer object). You'll use this form inside models.

print(fm(relu_out))
assert torch.equal(relu_out, functional_relu) and torch.equal(functional_relu, nn_relu)

In [ ]:
# The same pattern works for the others: torch.sigmoid / F.sigmoid / nn.Sigmoid(), and so on
print("sigmoid:\n", fm(torch.sigmoid(A), 3), "\n")
print("tanh:\n", fm(torch.tanh(A), 3), "\n")
print("softmax over each row:\n", fm(torch.softmax(A, dim=1), 3))  # softmax always needs a dim

### Exercise 4: Plot the activation functions
 **Before you write any code**, sketch on paper what you expect each function to look like between −5 and 5.

Then fill in each activation applied to `x`.

In [ ]:
x = torch.linspace(-5, 5, 100)  # 100 evenly spaced points from -5 to 5

activations = {
    "ReLU": None,
    "Sigmoid": None,
    "Tanh": None,
    "Softmax": None,  # x is 1-D, so use dim=0
}

In [ ]:
# Check your work
assert torch.allclose(activations["ReLU"], x.clamp(min=0))
assert torch.allclose(activations["Sigmoid"], 1 / (1 + torch.exp(-x)))
assert torch.allclose(activations["Tanh"], (torch.exp(x) - torch.exp(-x)) / (torch.exp(x) + torch.exp(-x)), atol=1e-6)
assert torch.allclose(activations["Softmax"].sum(), torch.tensor(1.0))
print("All checks passed")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, y) in zip(axes, activations.items()):
    ax.plot(x, y)
    ax.set_title(name)
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5)
    ax.grid(alpha=0.3)
plt.show()

**Questions to think about**
1. Did the plots match your sketches? Which functions can output negative numbers?
2. Sigmoid and tanh flatten out for large $|x|$. What does that mean for their **slope** (derivative) there? Hint: compare with $s'(0)$ from Exercise 3d. This is the *vanishing gradient* problem, and it's one reason ReLU is the default for hidden layers.
3. Why does the softmax plot look different? Softmax isn't applied to each value on its own. Every output depends on **all 100** inputs, because they must sum to 1. That's why each value is small, and why most of the probability goes to the largest inputs near $x = 5$.

# 5. Saving and loading
You will often need to save and load tensors and models. This section shows how to do both in `PyTorch`.

Saving is simple: `torch.save(value, file_path)`. It works for a single tensor or for any Python object that contains tensors, such as a dictionary. In the next notebook, you'll save a trained model's weights this way.

### Exercise 5
Create a 2×3 tensor of random values with dtype `torch.float16` and save it as `'example_tensor.pt'`.

In [ ]:
saved_tensor = None
# save it here

Now load the tensor back in with `torch.load(file_path)` and check that the loaded values equal the saved ones.

In [ ]:
loaded_tensor = None

In [ ]:
# Check your work
assert saved_tensor.shape == (2, 3) and saved_tensor.dtype == torch.float16
assert torch.equal(saved_tensor, loaded_tensor)
print("All checks passed")

Congratulations you finished the task, submit this deliverable through Canvas.

### Cheat sheet
| Task | PyTorch |
|---|---|
| create | `torch.tensor`, `zeros`, `ones`, `rand`, `randn`, `arange`, `zeros_like` |
| inspect | `.shape`, `.ndim`, `.numel()`, `.dtype` |
| NumPy | `torch.from_numpy(a)`, `t.numpy()` |
| reshape | `reshape`, `view`, `flatten(start_dim=1)`, `squeeze`, `unsqueeze`, `permute`, `transpose` |
| combine | `torch.cat` (existing dim), `torch.stack` (new dim) |
| multiply | `*` (element-wise), `@` (matrix), `torch.dot` (vectors) |
| reduce | `sum`, `mean`, `max`, `argmax` with `dim=` and `keepdim=` |
| activations | `torch.relu`, `torch.sigmoid`, `torch.tanh`, `torch.softmax(x, dim=...)` |
| linear layer | `nn.Linear(in, out)`, which computes `x @ W.T + b` |
| save / load | `torch.save(obj, path)`, `torch.load(path)` |